# Baseline Analysis

In [7]:
import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

In [8]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [9]:
from social_groups.analysis.definitions import defs

baseline_frame = defs.load_fn().load_asset_value("baseline")

2026-03-06 10:54:24 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/baseline.parquet using PolarsParquetIOManager...


In [10]:
baseline_frame.head()

id,run_id,question_id,phoenix_span_url,run_identifier,final_answer,original_question_id,category,question,answer_string,model_name
i64,i64,i64,str,str,str,i64,str,str,str,str
30360,40,61,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",6682,"""health""","""Q: A 22-year-old male presents…","""H""","""Qwen/Qwen3-4B"""
30361,40,65,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",3249,"""biology""","""Q: How does the term ""growth"" …","""D""","""Qwen/Qwen3-4B"""
30362,40,66,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",2626,"""psychology""","""Q: A non-custodial parent asks…","""J""","""Qwen/Qwen3-4B"""
30363,40,35,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",8107,"""math""","""Q: A telephone executive instr…","""C""","""Qwen/Qwen3-4B"""
30364,40,69,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",3100,"""biology""","""Q: What are the difficulties i…","""D""","""Qwen/Qwen3-4B"""


### Number of unparsable answers

In [11]:
(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .group_by("model_name")
    .agg(
        no_null=pl.col(AnalysisColumn.parsed_answer.value)
        .str.starts_with("___")
        .not_()
        .sum(),
        null_percentage=(
            pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
            * 100
        ).round(2),
    )
)

model_name,no_null,null_percentage
str,u32,f64
"""Qwen/Qwen3-14B""",93,7.0
"""Qwen/Qwen3-0.6B""",90,10.0
"""Qwen/Qwen3-4B""",88,12.0


### Accuracy per Model

In [12]:
(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.35
"""Qwen/Qwen3-4B""",0.59
"""Qwen/Qwen3-14B""",0.64
